In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [41]:
df = pd.read_csv(
    "../data/land_acquisition_india_200k_minimal/land_acquisition_india_200k.csv"
)

df.head()

,project_id,project_type,land_area_hectares,number_of_affected_families,compensation_status,approval_timeline_days,legal_disputes_count,possession_status,rehabilitation_progress_pct,stakeholder_responsiveness,historical_performance_score,administrator_id,manager,location,altitude_m,latitude,longitude,delay_status,delay_days,risk_score
0,LA-IND-0000001,Highway,41.58,9,Assessment/Processing,247,1,Acquired,63.5,Low,61.9,ADM-0290,MGR-05091,"Nainital, Uttarakhand",1612.9,30.0668,78.640313,Delayed,248,65.2
1,LA-IND-0000002,Railway,53.72,8,Paid in Full,223,1,Pending,67.3,Low,45.2,ADM-0350,MGR-10763,"Salem, Tamil Nadu",433.6,11.1271,77.155395,Delayed,186,65.7
2,LA-IND-0000003,Power,15.50,5,Not Started,172,1,Acquired,69.7,Medium,53.6,ADM-0040,MGR-05135,"Nainital, Uttarakhand",1512.9,30.0668,78.864017,On Time,55,58.5
3,LA-IND-0000004,Irrigation,31.00,11,Disputed,195,1,Partially Acquired,55.5,Medium,70.6,ADM-1065,MGR-05648,"Kamrup Metropolitan, Assam",259.6,26.2006,92.495593,On Time,47,60.8
4,LA-IND-0000005,Highway,110.35,123,Disputed,394,1,Acquired,80.0,Medium,53.9,ADM-1090,MGR-06966,"Khordha, Odisha",362.2,20.9517,85.049115,Delayed,248,66.4


In [42]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 200000
Columns: 20


### Validate columns

In [43]:
expected_columns = [
    "project_id",
    "project_type",
    "land_area_hectares",
    "number_of_affected_families",
    "compensation_status",
    "approval_timeline_days",
    "legal_disputes_count",
    "possession_status",
    "rehabilitation_progress_pct",
    "stakeholder_responsiveness",
    "historical_performance_score",
    "administrator_id",
    "manager",
    "location",
    "altitude_m",
    "latitude",
    "longitude",
    "delay_status",
    "delay_days",
    "risk_score"
]

missing_columns = set(expected_columns) - set(df.columns)
extra_columns = set(df.columns) - set(expected_columns)

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

Missing columns: set()
Extra columns: set()


### Duplicate handling

In [44]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate project IDs:", df["project_id"].duplicated().sum())

Duplicate rows: 0
Duplicate project IDs: 0


### Standardize categorical values

In [45]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].astype("string").str.strip()

C:\Users\pankaj\AppData\Local\Temp\ipykernel_29928\2617809793.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


In [46]:
for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(20))


project_id
project_id
LA-IND-0000001    1
LA-IND-0000002    1
LA-IND-0000003    1
LA-IND-0000004    1
LA-IND-0000005    1
LA-IND-0000006    1
LA-IND-0000007    1
LA-IND-0000008    1
LA-IND-0000009    1
LA-IND-0000010    1
LA-IND-0000011    1
LA-IND-0000012    1
LA-IND-0000013    1
LA-IND-0000014    1
LA-IND-0000015    1
LA-IND-0000016    1
LA-IND-0000017    1
LA-IND-0000018    1
LA-IND-0000019    1
LA-IND-0000020    1
Name: count, dtype: Int64

project_type
project_type
Highway                 50009
Railway                 34310
Irrigation              24104
Other Infrastructure    19856
Urban Transport         17994
Power                   17779
Water Supply            14040
Industrial              13924
Airport                  7984
Name: count, dtype: Int64

compensation_status
compensation_status
Paid in Full             63940
Partially Paid           55906
Assessment/Processing    38031
Disputed                 21988
Not Started              20135
Name: count, dtype: Int64

posse

### Check invalid numerical values

In [47]:
numeric_columns = df.select_dtypes(include=np.number).columns

print(df[numeric_columns].describe().T)

                                 count        mean         std        min  \
land_area_hectares            200000.0   64.393192   65.206044   0.950000   
number_of_affected_families   200000.0   42.022530   71.194418   1.000000   
approval_timeline_days        200000.0  243.967505   74.268002  15.000000   
legal_disputes_count          200000.0    1.597280    1.497310   0.000000   
rehabilitation_progress_pct   200000.0   67.336788   19.998181   0.000000   
historical_performance_score  200000.0   58.729159   14.279746  10.000000   
altitude_m                    200000.0  448.412172  510.092214   0.000000   
latitude                      200000.0   22.269698    5.903773  10.566700   
longitude                     200000.0   80.160722    5.919266  69.475257   
delay_days                    200000.0  124.456540  142.911714   0.000000   
risk_score                    200000.0   57.250010   12.862929   2.900000   

                                     25%         50%         75%          m

In [48]:
print("Negative land area:",
      (df["land_area_hectares"] < 0).sum())

print("Negative affected families:",
      (df["number_of_affected_families"] < 0).sum())

print("Negative legal disputes:",
      (df["legal_disputes_count"] < 0).sum())

print("Invalid rehabilitation percentage:",
      ((df["rehabilitation_progress_pct"] < 0) |
       (df["rehabilitation_progress_pct"] > 100)).sum())

print("Invalid latitude:",
      ((df["latitude"] < -90) |
       (df["latitude"] > 90)).sum())

print("Invalid longitude:",
      ((df["longitude"] < -180) |
       (df["longitude"] > 180)).sum())

Negative land area: 0
Negative affected families: 0
Negative legal disputes: 0
Invalid rehabilitation percentage: 0
Invalid latitude: 0
Invalid longitude: 0


### Handle missing values

In [49]:
missing = df.isnull().sum().sort_values(ascending=False)

print(missing[missing > 0])

Series([], dtype: int64)


### Remove obvious outcome/leakage columns

In [50]:
TARGET = "delay_status"

DROP_COLUMNS = [
    "project_id",
    "administrator_id",
    "manager",
    "location",
    "delay_days",
    "risk_score"
]

X = df.drop(columns=DROP_COLUMNS + [TARGET])
y = df[TARGET]

In [51]:
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer



### Train/test split

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (160000, 13)
X_test: (40000, 13)


### Fit imputer

In [53]:
numerical_features=X.select_dtypes(include=np.number).columns.tolist()

imputer = SimpleImputer(strategy="median")

X_train[numerical_features] = imputer.fit_transform(
    X_train[numerical_features]
)

X_test[numerical_features] = imputer.transform(
    X_test[numerical_features]
)

In [54]:
joblib.dump(
    imputer,
    "../model/imputer.pkl"
)

print("imputer.pkl saved")

imputer.pkl saved


In [55]:
X.head()

,project_type,land_area_hectares,number_of_affected_families,compensation_status,approval_timeline_days,legal_disputes_count,possession_status,rehabilitation_progress_pct,stakeholder_responsiveness,historical_performance_score,altitude_m,latitude,longitude
0,Highway,41.58,9,Assessment/Processing,247,1,Acquired,63.5,Low,61.9,1612.9,30.0668,78.640313
1,Railway,53.72,8,Paid in Full,223,1,Pending,67.3,Low,45.2,433.6,11.1271,77.155395
2,Power,15.50,5,Not Started,172,1,Acquired,69.7,Medium,53.6,1512.9,30.0668,78.864017
3,Irrigation,31.00,11,Disputed,195,1,Partially Acquired,55.5,Medium,70.6,259.6,26.2006,92.495593
4,Highway,110.35,123,Disputed,394,1,Acquired,80.0,Medium,53.9,362.2,20.9517,85.049115


In [56]:
X.dtypes

project_type                     string
land_area_hectares              float64
number_of_affected_families       int64
compensation_status              string
approval_timeline_days            int64
legal_disputes_count              int64
possession_status                string
rehabilitation_progress_pct     float64
stakeholder_responsiveness       string
historical_performance_score    float64
altitude_m                      float64
latitude                        float64
longitude                       float64
dtype: object

### Categorical missing values

In [57]:
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

X_train[categorical_features] = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

X_test[categorical_features] = categorical_imputer.transform(
    X_test[categorical_features]
)

In [58]:
joblib.dump(
    categorical_imputer,
    "../model/categorical_imputer.pkl"
)

print("categorical_imputer.pkl saved")

categorical_imputer.pkl saved


### Save raw input columns

In [59]:
raw_input_columns = X.columns.tolist()

joblib.dump(
    raw_input_columns,
    "../model/raw_input_columns.pkl"
)

['../model/raw_input_columns.pkl']

In [60]:
print(raw_input_columns)

['project_type', 'land_area_hectares', 'number_of_affected_families', 'compensation_status', 'approval_timeline_days', 'legal_disputes_count', 'possession_status', 'rehabilitation_progress_pct', 'stakeholder_responsiveness', 'historical_performance_score', 'altitude_m', 'latitude', 'longitude']


### Encoding

In [61]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_cat = encoder.transform(
    X_test[categorical_features]
)

In [65]:
joblib.dump(
    encoder,
    "../model/encoder.pkl"
)

['../model/encoder.pkl']

### Create final feature columns

In [62]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

feature_columns = (
    numerical_features +
    encoded_columns.tolist()
)

In [63]:
joblib.dump(
    feature_columns,
    "../model/featurecolumn.pkl"
)

['../model/featurecolumn.pkl']

In [64]:
df.to_csv(
    "../data/processed/cleaned_data.csv",
    index=False
)